I. Data Cleaning

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('D:/Formation Analyse de données/projet/projet1/afrimarket_dataset_senior.csv')
df.describe()
print(df.head())

  id_commande date_commande id_client        ville     categorie  \
0   CMD100000    2025-12-11     C0379  Brazzaville          Mode   
1   CMD100001    2025-07-27     C1827   Libreville        Beauté   
2   CMD100002    2025-07-08     C0119      Abidjan  electronique   
3   CMD100003    2025-08-20     C1436   Libreville  electronique   
4   CMD100004    2025-11-28     C0003   Libreville  Électronique   

               nom_produit  prix_unitaire  quantite  remise  cout_livraison  \
0           Produit_Mode_9          49.72         3    0.07            2.45   
1         Produit_Beauté_3          37.18         2    0.10            6.89   
2  Produit_Électronique_36         315.60         0    0.04            5.60   
3  Produit_Électronique_29         143.25         0    0.03            5.96   
4  Produit_Électronique_45         498.71         2    0.02            5.16   

          methode_paiement canal_marketing  cout_marketing statut_commande  
0             Mobile Money           Em

In [3]:
df["date_commande"]=pd.to_datetime(df["date_commande"])


In [4]:
df["date_commande"].dtype

dtype('<M8[us]')

In [5]:
df["date_commande"].dt.strftime("%Y-%m-%d").head()

0    2025-12-11
1    2025-07-27
2    2025-07-08
3    2025-08-20
4    2025-11-28
Name: date_commande, dtype: str

In [6]:
df["date_commande"].dtype

dtype('<M8[us]')

In [ ]:
pd.set_option('display.max_rows', None)
df["categorie"]

0                Mode
1              Beauté
2        electronique
3        electronique
4        Électronique
5              Maison
6                Mode
7              Beauté
8                Mode
9        Électronique
10       Électronique
11       electronique
12               Mode
13               Mode
14       Électronique
15               Mode
16               Mode
17             Maison
18               Mode
19             Maison
20             Maison
21             Maison
22       electronique
23       Électronique
24       Électronique
25             Beauté
26             Maison
27             Beauté
28       electronique
29             Beauté
30             Maison
31             Maison
32             Maison
33               Mode
34       Électronique
35             Beauté
36             Beauté
37               Mode
38             Beauté
39               Mode
40             Maison
41             Beauté
42               Mode
43               Mode
44             Maison
45        

In [7]:
df["categorie"].unique()

<StringArray>
['Mode', 'Beauté', 'electronique', 'Électronique', 'Maison']
Length: 5, dtype: str

méthode ULTRA PRO que les Data Analysts utilisent pour détecter automatiquement des catégories non uniformisées
Détecter les variantes cachées (méthode entreprise)

👉 On compare :

version originale

version normalisée


In [12]:
df["categorie_clean_test"]=(df["categorie"].str.lower().str.normalize("NFKD").str.encode("ascii", errors="ignore").str.decode("utf-8"))

2️⃣ Voir les différences entre les deux

In [13]:
df[["categorie","categorie_clean_test"]].drop_duplicates()


,categorie,categorie_clean_test
0,Mode,mode
1,Beaute,beaute
2,Electronique,electronique
5,Maison,maison


3️⃣ Détection automatique des catégories suspectes

In [14]:
df.groupby("categorie_clean_test")["categorie"].unique()

categorie_clean_test
beaute                [Beaute]
electronique    [Electronique]
maison                [Maison]
mode                    [Mode]
Name: categorie, dtype: object

4️⃣ Ensuite tu peux corriger définitivement

In [24]:
df[["categorie","categorie_clean_test"]].head()



,categorie,categorie_clean_test
0,Mode,mode
1,Beaute,beaute
2,Electronique,electronique
3,Electronique,electronique
4,Electronique,electronique


In [25]:
df["categorie"]=df["categorie_clean_test"].str.capitalize()

In [26]:
df.drop(columns="categorie_clean_test", inplace=True)


●	Corriger les villes mal orthographiées

In [19]:
df["ville"].unique()

<StringArray>
['Brazzaville',  'Libreville',     'Abidjan',       'Dakar',        'Lomé',
     'Cotonou',    'Kinshasa',      'Douala',   'Kinshassa']
Length: 9, dtype: str

villes ne sont pas uniformisées non plus.

Je vois immédiatement un problème :

Kinshasa
Kinshassa   ❌ (faute d’orthographe)
Pour Python ce sont deux villes différentes.

In [27]:
df["ville"].value_counts()

ville
Kinshasa       2349
Abidjan        1886
Douala         1449
Dakar          1434
Lomé            735
Cotonou         677
Kinshassa       605
Libreville      493
Brazzaville     472
Name: count, dtype: int64

In [30]:
df["ville"]=df["ville"].replace({"Kinshassa":"Kinshasa" })

In [31]:
df["ville"].unique()

<StringArray>
['Brazzaville',  'Libreville',     'Abidjan',       'Dakar',        'Lomé',
     'Cotonou',    'Kinshasa',      'Douala']
Length: 8, dtype: str

In [32]:
df["remise"]

0        0.07
1        0.10
2        0.04
3        0.03
4        0.02
         ... 
10095    0.10
10096    0.12
10097   -0.10
10098    0.20
10099    0.30
Name: remise, Length: 10100, dtype: float64

Gérer les remises négatives” veut dire :
traiter les cas où la colonne remise contient des valeurs < 0 (ce qui est souvent une anomalie dans les données).

In [34]:
df[df["remise"]<0]

,id_commande,date_commande,id_client,ville,categorie,nom_produit,prix_unitaire,quantite,remise,cout_livraison,methode_paiement,canal_marketing,cout_marketing,statut_commande
6,CMD100006,2025-07-24,C0183,Abidjan,Mode,Produit_Mode_3,23.04,3,-0.1,3.59,Virement,Instagram Ads,9.23,Livrée
23,CMD100023,2025-09-06,C1560,Kinshasa,Electronique,Produit_Électronique_41,370.82,3,-0.1,2.56,Carte,Instagram Ads,8.73,Livrée
33,CMD100033,2025-07-09,C0145,Dakar,Mode,Produit_Mode_43,41.26,1,-0.1,4.19,Virement,Google Ads,6.26,Livrée
38,CMD100038,2025-08-18,C0071,Douala,Beaute,Produit_Beauté_5,18.17,1,-0.1,2.73,Mobile Money,Instagram Ads,11.73,Livrée
54,CMD100054,2025-11-19,C0339,Kinshasa,Electronique,Produit_Électronique_17,443.99,1,-0.1,3.35,Mobile Money,Influenceur,4.65,retournée
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10068,CMD100201,2025-11-16,C0089,Dakar,Electronique,Produit_Électronique_9,-50.00,2,-0.1,8.72,Carte,Influenceur,15.19,retournée
10071,CMD107405,2025-07-19,C0141,Abidjan,Maison,Produit_Maison_37,83.82,2,-0.1,4.16,Mobile Money,Google Ads,7.04,Livrée
10077,CMD101496,2025-08-20,C0512,Douala,Electronique,Produit_Électronique_13,339.74,2,-0.1,5.43,Virement,Instagram Ads,5.25,Livrée
10083,CMD109681,2025-10-04,C0356,Kinshasa,Maison,Produit_Maison_37,139.00,2,-0.1,7.06,Paiement à la livraison,Instagram Ads,10.49,Livrée


In [35]:
(df["remise"]<0).sum()


np.int64(614)

méthode A: remplaer par 0
df.loc[ ... , ... ]

👉 .loc sert à sélectionner des lignes et des colonnes.

Structure :

df.loc[lignes, colonnes]
Dans ton cas :

lignes = df["remise"] < 0
colonnes = "remise"


Donc :

👉 sélectionne uniquement les lignes où la remise est négative, dans la colonne remise.
= 0

👉 Remplace les valeurs sélectionnées par 0.

In [36]:
df.loc[df["remise"]<0,"remise"]=0

In [37]:
(df["remise"]<0).sum()

np.int64(0)

●	Gérer les prix aberrants
détecter et traiter les prix qui semblent anormaux dans ton dataset.

Un prix aberrant = un prix qui sort du comportement normal.

In [38]:
df["prix_unitaire"].describe()

count    10100.000000
mean       140.493353
std        172.031607
min        -50.000000
25%         28.057500
50%         52.975000
75%        201.885000
max        915.860000
Name: prix_unitaire, dtype: float64

Prix négatifs détectés :-50
Prix très élevés (potentiels outliers) : 915.860000
Mais la médiane est seulement :52.97
👉 Ça veut dire qu’il y a des valeurs très éloignées de la majorité.
3️⃣ Méthode PRO — Calculer les limites IQR
Tout ce qui dépasse ~462 est probablement aberrant.

In [42]:
Q1=df["prix_unitaire"].quantile(0.25)
Q3=df["prix_unitaire"].quantile(0.75)
IQR=Q3-Q1
lower_bound=Q1-1.5*IQR
upper_bound=Q3+1.5*IQR
print(lower_bound, upper_bound )
outliers=df[(df["prix_unitaire"]<lower_bound) | (df["prix_unitaire"]>upper_bound)]
print(outliers)

-232.68374999999997 462.62624999999997
      id_commande date_commande id_client        ville     categorie  \
4       CMD100004    2025-11-28     C0003   Libreville  Electronique   
9       CMD100009    2025-12-17     C0148        Dakar  Electronique   
11      CMD100011    2025-09-29     C0136      Cotonou  Electronique   
14      CMD100014    2025-09-22     C1589      Abidjan  Electronique   
60      CMD100060    2025-07-10     C1192      Abidjan  Electronique   
...           ...           ...       ...          ...           ...   
10018   CMD103281    2025-07-01     C0354     Kinshasa  Electronique   
10026   CMD106712    2025-12-23     C0126       Douala  Electronique   
10053   CMD102428    2025-08-19     C1255  Brazzaville  Electronique   
10073   CMD103755    2025-11-21     C0069      Abidjan  Electronique   
10089   CMD107864    2025-10-09     C0077     Kinshasa  Electronique   

                   nom_produit  prix_unitaire  quantite  remise  \
4      Produit_Électronique_4

In [43]:
df["prix_unitaire"]=df["prix_unitaire"].clip(lower=0, upper=upper_bound)

In [44]:
df["prix_unitaire"].describe()

count    10100.000000
mean       137.415476
std        154.374639
min          0.000000
25%         28.057500
50%         52.975000
75%        201.885000
max        462.626250
Name: prix_unitaire, dtype: float64

●	Supprimer ou traiter les quantités nulles

In [45]:
df["quantite"].describe()

count    10100.000000
mean         1.866040
std          0.920966
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max          3.000000
Name: quantite, dtype: float64

In [46]:
(df["quantite"]==0).sum()

np.int64(608)

In [47]:
df[df["quantite"]==0]

,id_commande,date_commande,id_client,ville,categorie,nom_produit,prix_unitaire,quantite,remise,cout_livraison,methode_paiement,canal_marketing,cout_marketing,statut_commande
2,CMD100002,2025-07-08,C0119,Abidjan,Electronique,Produit_Électronique_36,315.60000,0,0.04,5.60,Carte,Google Ads,7.30,Livrée
3,CMD100003,2025-08-20,C1436,Libreville,Electronique,Produit_Électronique_29,143.25000,0,0.03,5.96,Mobile Money,Google Ads,5.20,Livrée
35,CMD100035,2025-10-30,C0389,Dakar,Beaute,Produit_Beauté_23,21.56000,0,0.15,6.61,Paiement à la livraison,Instagram Ads,14.68,Livrée
45,CMD100045,2025-09-19,C0299,Brazzaville,Maison,Produit_Maison_4,122.17000,0,0.06,6.67,Mobile Money,Email,0.78,Livrée
46,CMD100046,2025-11-08,C0095,Brazzaville,Electronique,Produit_Électronique_44,170.48000,0,0.02,6.91,Carte,Google Ads,7.75,retournée
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10073,CMD103755,2025-11-21,C0069,Abidjan,Electronique,Produit_Électronique_1,462.62625,0,0.29,4.08,Mobile Money,Instagram Ads,7.66,Livrée
10076,CMD100304,2025-09-20,C1453,Kinshasa,Mode,Produit_Mode_20,49.59000,0,0.27,7.73,Virement,Google Ads,3.78,retournée
10088,CMD107474,2025-08-03,C1480,Kinshasa,Beaute,Produit_Beauté_36,26.14000,0,0.18,9.00,Paiement à la livraison,Instagram Ads,6.41,Livrée
10096,CMD108770,2025-12-27,C0009,Kinshasa,Mode,Produit_Mode_37,21.12000,0,0.12,8.91,Virement,Instagram Ads,10.46,Livrée


In [ ]:
Supprimer les quantités nulles (le plus courant)

In [48]:
df=df[df["quantite"]>0]

In [49]:
(df["quantite"]==0).sum()

np.int64(0)

●	Gérer les doublons
Un doublon = deux lignes (ou plus) identiques.
Étape 1 — Détecter les doublons

In [50]:
df.duplicated().sum()

np.int64(92)

Afficher les doublons

In [51]:
df[df.duplicated() ]

,id_commande,date_commande,id_client,ville,categorie,nom_produit,prix_unitaire,quantite,remise,cout_livraison,methode_paiement,canal_marketing,cout_marketing,statut_commande
10000,CMD102576,2025-07-31,C0924,Kinshasa,Mode,Produit_Mode_14,13.63,1,0.24,7.42,Paiement à la livraison,Google Ads,7.62,retournée
10001,CMD102940,2025-09-24,C0039,Brazzaville,Mode,Produit_Mode_45,44.07,1,0.10,9.77,Carte,Google Ads,7.80,Livrée
10002,CMD107540,2025-09-21,C1880,Abidjan,Electronique,Produit_Électronique_14,0.00,1,0.04,7.34,Mobile Money,Google Ads,4.76,Livrée
10003,CMD104223,2025-12-12,C1305,Cotonou,Mode,Produit_Mode_28,32.56,2,0.26,8.42,Paiement à la livraison,Instagram Ads,8.40,Livrée
10005,CMD103856,2025-09-15,C0012,Abidjan,Beaute,Produit_Beauté_22,8.94,1,0.21,5.35,Carte,Instagram Ads,5.50,Livrée
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10093,CMD104687,2025-10-17,C0394,Kinshasa,Mode,Produit_Mode_35,0.00,3,0.20,5.96,Mobile Money,Email,0.70,Livrée
10094,CMD101287,2025-10-27,C0359,Douala,Electronique,Produit_Électronique_39,239.26,3,0.10,4.32,Paiement à la livraison,Email,1.84,retournée
10095,CMD104520,2025-10-07,C0176,Brazzaville,Beaute,Produit_Beauté_44,24.82,1,0.10,3.75,Carte,Instagram Ads,8.06,Livrée
10098,CMD109850,2025-09-22,C0292,Abidjan,Beaute,Produit_Beauté_48,29.13,1,0.20,9.47,Paiement à la livraison,Instagram Ads,9.88,Livrée


Étape 2 — Vérifier selon certaines colonnes

In [53]:
df=df.drop_duplicates()

In [55]:
df.shape

(9400, 14)

In [56]:
df[df.duplicated() ]

,id_commande,date_commande,id_client,ville,categorie,nom_produit,prix_unitaire,quantite,remise,cout_livraison,methode_paiement,canal_marketing,cout_marketing,statut_commande


Les doublons ont été supprimés afin d’éviter une surreprésentation
de certaines transactions dans l’analyse.

👉 “Uniformiser les statuts” veut dire :

rendre toutes les valeurs de la colonne statut cohérentes et identiques pour éviter les variantes.

In [61]:
df["statut_commande"].unique()

<StringArray>
['Livrée', 'retournée', 'Annulée']
Length: 3, dtype: str

In [62]:
df["statut_commande"].value_counts(dropna=False )


statut_commande
Livrée       8452
retournée     765
Annulée       183
Name: count, dtype: int64

✅ Tes statuts sont déjà presque uniformisés (il n’y a que 3 valeurs).

Le seul “petit problème” visible : la casse (ex: retournée est en minuscule alors que les autres commencent par une majuscule).

In [63]:
df["statut_commande"]=df["statut_commande"].str.strip().str.capitalize()

In [64]:
df["statut_commande"].unique()

<StringArray>
['Livrée', 'Retournée', 'Annulée']
Length: 3, dtype: str

In [65]:
df_clean=df

In [ ]:
df_clean

NameError: name 'df_clean' is not defined